# Beige Book Pipeline v2 (SDK-only)

End-to-end pipeline using only SDK functions — no local Qdrant, no local labeler, no custom renderer.

**Stages:**
1. Download Beige Book PDFs
2. Upload to a FileSet (skip if already done)
3. Run a single `QuestionPipeline`:
   - `FileSetSeedGenerator` — chunks every PDF as seeds
   - `ForwardLookingQuestionGenerator` — generates district-specific forecasting questions
   - `FileSetRAGLabeler(AFTER)` — resolves each question from a *later* Beige Book
   - `FileSetContextGenerator(BEFORE)` — enriches each question with *earlier* Beige Books
   - `QuestionRenderer(template)` — renders the final prompt
4. Prepare for training (filter, dedup, train/test split)
5. Train with SDK (GRPO/RL)
6. Evaluate on test split
7. Analyze results

In [ ]:
import os
import time
import requests
from datetime import datetime, timezone
from pathlib import Path

from lightningrod import (
    LightningRod,
    BinaryAnswerType,
    QdrantRAGLabeler,
    FileSetContextGenerator,
    FileSetSeedGenerator,
    ForwardLookingQuestionGenerator,
    QuestionPipeline,
    QuestionRenderer,
    TemporalConstraint,
)
from lightningrod._generated.models import (
    FileSetMetadataSchemaInput,
    MetadataFieldDefinitionInput,
    MetadataFieldType,
)

lr = LightningRod(
    api_key=os.getenv("LR_PROD_API_KEY"),
    base_url="https://api.lightningrod.ai/api/public/v1",
)

ImportError: cannot import name 'FileSetContextGenerator' from 'lightningrod' (/home/pwilczewski/ML/lightningrod-python-sdk/.venv/lib/python3.10/site-packages/lightningrod/__init__.py)

## 1. Download Beige Book PDFs

In [2]:
BEIGE_BOOK_DATES = [
    "20240117", "20240306", "20240417", "20240529", "20240717", "20240904", "20241023", "20241204",
    "20250115", "20250305", "20250423", "20250604", "20250716", "20250903", "20251015", "20251126",
    "20260114", "20260304",
]

BASE_URL = "https://www.federalreserve.gov/monetarypolicy/files/BeigeBook_{date}.pdf"
pdf_dir = Path("files")
pdf_dir.mkdir(exist_ok=True)

for date_str in BEIGE_BOOK_DATES:
    out_path = pdf_dir / f"BeigeBook_{date_str}.pdf"
    if out_path.exists():
        continue
    r = requests.get(BASE_URL.format(date=date_str))
    r.raise_for_status()
    out_path.write_bytes(r.content)
    print(f"downloaded: {out_path.name}")

print(f"{len(list(pdf_dir.glob('BeigeBook_*.pdf')))} PDFs ready")

downloaded: BeigeBook_20240117.pdf
downloaded: BeigeBook_20240306.pdf
downloaded: BeigeBook_20240417.pdf
downloaded: BeigeBook_20240529.pdf
downloaded: BeigeBook_20240717.pdf
downloaded: BeigeBook_20240904.pdf
downloaded: BeigeBook_20241023.pdf
downloaded: BeigeBook_20241204.pdf
downloaded: BeigeBook_20250115.pdf
downloaded: BeigeBook_20250305.pdf
downloaded: BeigeBook_20250423.pdf
downloaded: BeigeBook_20250604.pdf
downloaded: BeigeBook_20250716.pdf
downloaded: BeigeBook_20250903.pdf
downloaded: BeigeBook_20251015.pdf
downloaded: BeigeBook_20251126.pdf
downloaded: BeigeBook_20260114.pdf
downloaded: BeigeBook_20260304.pdf
18 PDFs ready


## 2. FileSet setup

Create a new FileSet and upload all PDFs with `file_date` set from the filename date — this is what powers `TemporalConstraint.BEFORE` / `AFTER` filtering.

Skip this section and set `FILESET_ID` directly if the FileSet already exists.

In [3]:
schema = FileSetMetadataSchemaInput(fields=[
    MetadataFieldDefinitionInput(
        name="date", field_type=MetadataFieldType.STRING, required=True,
    ),
])
fileset = lr.filesets.create(
    name="Beige Book Reports",
    description="Federal Reserve Beige Book PDFs 2024-2026",
    metadata_schema=schema,
)
FILESET_ID = fileset.id
print(f"Created FileSet: {FILESET_ID}")

for pdf_path in sorted(pdf_dir.glob("BeigeBook_*.pdf")):
    date_str = pdf_path.stem.replace("BeigeBook_", "")
    file_date = datetime.strptime(date_str, "%Y%m%d").replace(tzinfo=timezone.utc)
    lr.filesets.files.upload(
        file_set_id=FILESET_ID,
        file_path=pdf_path,
        metadata={"date": date_str},
        file_date=file_date,
    )
    print(f"uploaded: {pdf_path.name}")

# Poll until all files are ACTIVE
print("Waiting for files to become ACTIVE...")
while True:
    files = lr.filesets.files.list(file_set_id=FILESET_ID).files
    statuses = [f.status for f in files]
    if all(s == "ACTIVE" for s in statuses):
        print(f"All {len(files)} files ACTIVE")
        break
    pending = sum(1 for s in statuses if s != "ACTIVE")
    print(f"  {pending}/{len(files)} still processing...")
    time.sleep(15)

Created FileSet: 9e82e919-4f2c-4c9d-b7e5-5e23c55b76d1
uploaded: BeigeBook_20240117.pdf
uploaded: BeigeBook_20240306.pdf
uploaded: BeigeBook_20240417.pdf
uploaded: BeigeBook_20240529.pdf
uploaded: BeigeBook_20240717.pdf
uploaded: BeigeBook_20240904.pdf
uploaded: BeigeBook_20241023.pdf
uploaded: BeigeBook_20241204.pdf
uploaded: BeigeBook_20250115.pdf
uploaded: BeigeBook_20250305.pdf
uploaded: BeigeBook_20250423.pdf
uploaded: BeigeBook_20250604.pdf
uploaded: BeigeBook_20250716.pdf
uploaded: BeigeBook_20250903.pdf
uploaded: BeigeBook_20251015.pdf
uploaded: BeigeBook_20251126.pdf
uploaded: BeigeBook_20260114.pdf
uploaded: BeigeBook_20260304.pdf
Waiting for files to become ACTIVE...
  5/18 still processing...
All 18 files ACTIVE


## 3. Build pipeline

In [ ]:
answer_type = BinaryAnswerType()

template = (
    "You are an expert economic forecaster analyzing Federal Reserve Beige Book reports. "
    "You will be given a question about a future economic outcome, a prior Beige Book excerpt "
    "(context only — the labeled outcome is determined by a later report), and historical context "
    "from past Beige Book reports. Predict the probability that the outcome will occur.\n\n"
    "TODAY'S DATE: {question_date}\n\n"
    "QUESTION:\n{question_text}\n\n"
    "RESOLUTION CRITERIA:\n{resolution_criteria}\n\n"
    "PRIOR REPORT EXCERPT (context only — answer comes from a future report):\n{seed_text}\n\n"
    "HISTORICAL CONTEXT (past Beige Book excerpts):\n{context}\n\n"
    "Think step by step, then output your prediction.\n\n"
    "ANSWER FORMAT:\n{answer_instructions}"
)

pipeline = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=FILESET_ID,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        questions_per_seed=5,
        answer_type=answer_type,
        instructions=(
            "Generate questions about whether specific economic outcomes will occur in the near future "
            "(decrease, increase, slow, accelerate, etc.). "
            "Ask about the outcome directly - e.g. 'Will loan nonperformance in Dallas decrease?' - "
            "NOT 'Will the next Beige Book report that...'. "
            "Do NOT use explicit dates, months, or years in the question or resolution criteria. "
            "Resolution criteria describe WHAT to look for (conditions for Yes/No/Undetermined), "
            "never WHICH document - the pipeline always provides the correct document. "
            "Never reference specific report dates, release dates, or months/years. "
            "Focus on district-specific topics (Dallas, St. Louis, Boston, etc.) and metrics "
            "that the Beige Book explicitly reports on. "
            "Resolution criteria MUST state: resolve Yes/No ONLY when the document explicitly "
            "addresses the topic in the relevant district section; if the topic is not reported "
            "or not mentioned, resolve as Undetermined (unverifiable). "
            "For increase/decrease/improvement questions: criteria MUST explicitly state that "
            "flat, stable, unchanged, held steady, 'about flat', or no change = No. "
            "Generate only questions that are likely to be explicitly addressed in the relevant "
            "district section - avoid topics that may be unreported."
        ),
        examples=[
            "Will loan nonperformance in the Dallas district decrease?",
            "Will employment growth in the Philadelphia district slow?",
            "Will manufacturing activity in the St. Louis district improve?",
        ],
        bad_examples=[
            "Will the next Beige Book report that loan nonperformance in Dallas has decreased? # REASON: uses report framing",
            "Will the Federal Reserve Bank of Dallas's October 2024 Beige Book report that X? # REASON: references specific dates",
            "Will economic activity in the Cleveland district change? # REASON: too vague (increase or decrease?)",
            "Will niche industry X in district Y improve? # REASON: unlikely to be explicitly addressed",
        ],
    ),
    labeler=QdrantRAGLabeler(
        file_set_id=FILESET_ID,
        temporal_constraint=TemporalConstraint.NEXT_DOCUMENT,  # resolve from future reports
        answer_type=answer_type,
        confidence_threshold=0.7,
    ),
    context_generators=[
        FileSetContextGenerator(
            file_set_id=FILESET_ID,
            temporal_constraint=TemporalConstraint.BEFORE,  # no lookahead leakage
        )
    ],
    renderer=QuestionRenderer(
        answer_type=answer_type,
        template=template,
    ),
)

## 4. Run pipeline

In [5]:
dataset = lr.transforms.run(
    pipeline,
    max_questions=2000,
    name="Beige Book SDK v1",
)
print(f"Dataset: {dataset.id}")
print(f"Rows: {dataset.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> API Error: submit transform job                                                                             │
│                                                                                                                 │
│  Failed to submit transform job: [ValidationError(loc=['body', 'config', 'QUESTION_PIPELINE', 'labeler'],       │
│  msg="Input tag 'FILESET_RAG_LABELER' found using 'config_type' does not match any of the expected tags:        │
│  <TransformType.WEB_SEARCH_LABELER: 'WEB_SEARCH_LABELER'>, <TransformType.QDRANT_RAG_LABELER:                   │
│  'QDRANT_RAG_LABELER'>, <TransformType.FILESET_DOCUMENT_LABELER: 'FILESET_DOCUMENT_LABELER'>,                   │
│  <TransformType.MOCK: 'MOCK'>", type_='union_tag_invalid', additional_properties={'input': {'file_set_id':      │
│  '9e82e919-4f2c-4c9d-b7e5-5e23c55b76d1', 'config_type': 'FILESET_RAG_LABELER', 'confidence_threshold': 0.7,     │
│  'answer_type': {'answer_type': 'BINARY', 'answer_format_instruction': "This is a binary yes/no question. You   │
│  are estimating the probability that the answer is 'Yes'. Provide your confidence as a value between 0          │
│  (definitely No) and 1 (definitely Yes). Provide your probability estimate for Yes as a decimal between 0 and   │
│  1. Provide your answer between <answer></answer> tags. Example: <answer>0.75</answer>",                        │
│  'labeler_instruction': "The answer should be ONLY '1', '0', or 'Undetermined'. '1' means yes, '0' means no,    │
│  and 'Undetermined' means the answer is not clear. Do not include any other text or explanation.",              │
│  'question_generation_instruction': "Generate binary forecasting questions about future events or outcomes      │
│  that are unresolved at the time of asking and will resolve to a clear, publicly verifiable Yes or No.\n\nEach  │
│  question MUST:\n- Have EXACTLY ONE binary answer: Yes or No\n- Be fully self-contained (all entities,          │
│  locations, dates included)\n- Refer to a clearly defined event or threshold with an explicit resolution date   │
│  or deadline\n- Describe an outcome plausibly reported in a major news headline or official release\n- Start    │
│  with words like 'Will', 'Is', 'Does', 'Has', 'Can', 'Did', or similar\n\nSTRICTLY DO NOT include:\n- Numeric   │
│  or continuous outcomes\n- Multiple-choice or categorical questions\n- Trivial, obscure, or low-impact          │
│  events\n- Vague language or ambiguous resolution criteria\n- Outcomes dependent on unpublished, proprietary,   │
│  or speculative data\n- Questions with more than two possible outcomes", 'reward_function_type':                │
│  'binary_brier', 'answer_parser_type': 'binary'}, 'temporal_constraint': 'NEXT_DOCUMENT', 'model':              │
│  'gemini-2.5-flash'}, 'ctx': {'discriminator': "'config_type'", 'tag': 'FILESET_RAG_LABELER', 'expected_tags':  │
│  "<TransformType.WEB_SEARCH_LABELER: 'WEB_SEARCH_LABELER'>, <TransformType.QDRANT_RAG_LABELER:                  │
│  'QDRANT_RAG_LABELER'>, <TransformType.FILESET_DOCUMENT_LABELER: 'FILESET_DOCUMENT_LABELER'>,                   │
│  <TransformType.MOCK: 'MOCK'>"}}), ValidationError(loc=['body', 'config', 'QUESTION_PIPELINE',                  │
│  'context_generators', 0, 'NewsContextGenerator', 'config_type'], msg="Input should be                          │
│  <TransformType.NEWS_CONTEXT_GENERATOR: 'NEWS_CONTEXT_GENERATOR'>", type_='literal_error',                      │
│  additional_properties={'input': 'FILESET_CONTEXT_GENERATOR', 'ctx': {'expected':                               │
│  "<TransformType.NEWS_CONTEXT_GENERATOR: 'NEWS_CONTEXT_GENERATOR'>"}}), ValidationError(loc=['body', 'config',  │
│  'QUESTION_PIPELINE', 'context_generators', 0, 'Qdrant

Exception: Failed to submit transform job: [ValidationError(loc=['body', 'config', 'QUESTION_PIPELINE', 'labeler'], msg="Input tag 'FILESET_RAG_LABELER' found using 'config_type' does not match any of the expected tags: <TransformType.WEB_SEARCH_LABELER: 'WEB_SEARCH_LABELER'>, <TransformType.QDRANT_RAG_LABELER: 'QDRANT_RAG_LABELER'>, <TransformType.FILESET_DOCUMENT_LABELER: 'FILESET_DOCUMENT_LABELER'>, <TransformType.MOCK: 'MOCK'>", type_='union_tag_invalid', additional_properties={'input': {'file_set_id': '9e82e919-4f2c-4c9d-b7e5-5e23c55b76d1', 'config_type': 'FILESET_RAG_LABELER', 'confidence_threshold': 0.7, 'answer_type': {'answer_type': 'BINARY', 'answer_format_instruction': "This is a binary yes/no question. You are estimating the probability that the answer is 'Yes'. Provide your confidence as a value between 0 (definitely No) and 1 (definitely Yes). Provide your probability estimate for Yes as a decimal between 0 and 1. Provide your answer between <answer></answer> tags. Exampl

## 5. Prepare for training

In [ ]:
# loading dataset to avoid re-running pipeline
dataset = lr.datasets.get("c0a082a6-1ba9-406f-ae26-1b77f966fd5c")

In [3]:
from lightningrod.training import prepare_for_training, SplitParams, FilterParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(),
    dedup=None,
    split=SplitParams(),
)
print(f"train: {train_dataset.num_rows}, test: {test_dataset.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 1820 samples                                                                                   │
│                                                                                                                 │
│    Filter:  Dropped 523 invalid → 1297 remain                                                                   │
│    Dedup:   Removed 381 duplicates (1297 → 916)                                                                 │
│      ('Will manufacturing activity in the Chicago district increas..., None): 16 samples → 1                    │
│      ('Will loan nonperformance in the Dallas district decrease?', None): 13 samples → 1                        │
│      ('Will manufacturing activity in the Minneapolis district imp..., None): 11 samples → 1                    │
│    Split:   Splits: 618 train | 184 test (0 dropped, no prediction_date)                                        │
│             114 train samples removed for leakage                                                               │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  Only 618 train samples remain after preparation. This is below the recommended minimum of +1000 for effective  │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_questions in lr.transforms.run() to generate more samples.                                  │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date to end_date) to capture more events.                     │
│                                                                                                                 │
│  Only 184 test samples remain after preparation. This is below the recommended minimum of +200 for reliable     │
│  evaluation.                                                                                                    │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Generate more samples overall — test samples come from the most recent portion of your date range.       │
│      • Ensure your seed generator date range extends close to the present so recent events appear in the test   │
│  set.                                                                                                           │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

train: 618, test: 184


## 6. Train with SDK (GRPO/RL)

In [4]:
from lightningrod import GRPOTrainingConfig

training_job = lr.training.create(
    GRPOTrainingConfig(
        base_model_id="openai/gpt-oss-120b",
        training_steps=20,
        batch_size=32,
        max_response_length=16384,
        lora_rank=32,
        num_rollouts=4,
        learning_rate=4e-5,
    ),
    dataset=train_dataset,
    name="beige-book-sdk-v1",
)
print(f"Job: {training_job.id}  status: {training_job.status}")

Job: abb4831b-ef5a-44d3-984c-8f5483aeff1f  status: STARTING


## 7. Evaluate on test split

In [ ]:
from lightningrod import EvalModel, training

eval_job = lr.evals.create(
    models=[EvalModel(model_id=training_job.model_id, label="base"),
            EvalModel(model_id=training_job.model_id, label="base")],
    dataset=test_lr,
)
print(f"Eval job: {eval_job.id}  status: {eval_job.status}")

In [23]:
from lightningrod import training

training_config = GRPOTrainingConfig(
        base_model_id="openai/gpt-oss-120b",
        training_steps=20,
        batch_size=32,
        max_response_length=16384,
        lora_rank=32,
        num_rollouts=4,
        learning_rate=4e-5,
)

eval_job = lr.evals.run(
    training_config,
    training_job,
    test_dataset,
)

training.print_eval(eval_job)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> API Error: create eval job                                                                                  │
│                                                                                                                 │
│  Failed to create eval job: Evaluation dataset validation failed (5 error(s)):                                  │
│  Row 41: missing 'correct_answer'                                                                               │
│  Row 72: missing 'correct_answer'                                                                               │
│  Row 75: missing 'correct_answer'                                                                               │
│  Row 89: missing 'correct_answer'                                                                               │
│  Too many invalid training rows (4/184), exceeds allowed ratio (HTTP 400)                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exception: Failed to create eval job: Evaluation dataset validation failed (5 error(s)):
Row 41: missing 'correct_answer'
Row 72: missing 'correct_answer'
Row 75: missing 'correct_answer'
Row 89: missing 'correct_answer'
Too many invalid training rows (4/184), exceeds allowed ratio (HTTP 400)